# Train, Compare, and Save the Winning Model

This notebook does all the model training for the deployed app. The Gradio app in `app.py` will not train any model.

The purpose of this notebook is to:

1. Load a small bundled sample of the airline passenger satisfaction dataset.
2. Train candidate classification pipelines on the same train/test split.
3. Compare their predictive performance.
4. Save the fitted pipelines and the model comparison results.

The saved `model.joblib` file will later be loaded by `app.py` to serve predictions for new passenger profiles.

## 1. Imports

In this section, we import the libraries needed for data loading, preprocessing, model training, evaluation and saving the final fitted model.

In [1]:
import time
from pathlib import Path

import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

## 2. Load the Bundled Sample

We load a small representative sample of the airline passenger satisfaction dataset.

The full training dataset is larger than what is needed for the deployed app. For this reason, a smaller sample is bundled with the app and used here for training, comparison and demonstration purposes.

In [2]:
HERE = Path.cwd()

DATA_PATH = HERE / "data" / "airline_passenger_satisfaction_sample.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(5000, 25)


,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,33389,84529,Male,disloyal Customer,36,Business travel,Business,2399,2,2,...,2,4,4,3,4,5,2,0,0.0,neutral or dissatisfied
1,19042,6190,Female,Loyal Customer,22,Personal Travel,Eco,409,0,1,...,2,2,2,3,1,3,2,0,0.0,satisfied
2,83312,124550,Male,Loyal Customer,45,Personal Travel,Eco,1276,2,5,...,3,3,2,5,4,4,3,0,3.0,neutral or dissatisfied
3,9364,59360,Male,Loyal Customer,27,Business travel,Business,2486,1,0,...,5,4,4,2,1,4,5,0,0.0,neutral or dissatisfied
4,54128,58500,Male,Loyal Customer,41,Business travel,Business,719,1,1,...,4,4,5,4,3,4,5,0,0.0,satisfied


## 3. One Train / Test Split, Used by Every Candidate

Every candidate algorithm is trained on the same training subset and evaluated on the same test subset. This makes the comparison fair.

The target variable is `satisfaction`. Technical identifier columns such as `id` and `Unnamed: 0` are removed because they do not provide useful predictive information.

In [3]:
TARGET = "satisfaction"

POSITIVE_LABEL = "satisfied"
NEGATIVE_LABEL = "neutral or dissatisfied"

# Remove technical identifier columns if they exist
drop_cols = [TARGET]

for col in ["Unnamed: 0", "id"]:
    if col in df.columns:
        drop_cols.append(col)

X = df.drop(columns=drop_cols)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=0
)

print("train", X_train.shape, "test", X_test.shape)
print("\nTarget distribution in training set:")
print(y_train.value_counts(normalize=True))

train (4000, 22) test (1000, 22)

Target distribution in training set:
satisfaction
neutral or dissatisfied    0.5665
satisfied                  0.4335
Name: proportion, dtype: float64


## 4. Define the Three Candidate Pipelines

Each candidate model is defined as a scikit-learn `Pipeline`.

The dataset contains both numerical and categorical features. For this reason, a common preprocessing step is used before each classifier:

- numerical features are imputed and scaled,
- categorical features are imputed and one-hot encoded.

The fitted preprocessing steps travel together with the classifier when the model is saved. This is important because the Gradio app must apply exactly the same transformations at prediction time.

The goal of this notebook is not extensive hyperparameter optimisation, but to follow the deployment pattern: train candidate models offline, compare them, and save the fitted pipelines for the app.

In [4]:
# Identify numerical and categorical features
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# Preprocessing for numerical variables
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Preprocessing for categorical variables
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
   ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

# Combine preprocessing steps
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

# Define candidate pipelines
candidates = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("clf", LogisticRegression(C=1.0, max_iter=1000, random_state=0)),
    ]),
    
    "K-Nearest Neighbors": Pipeline([
        ("preprocessor", preprocessor),
        ("clf", KNeighborsClassifier(n_neighbors=5, weights="uniform")),
    ]),
    
    "Decision Tree": Pipeline([
        ("preprocessor", preprocessor),
        ("clf", DecisionTreeClassifier(max_depth=8, random_state=0)),
    ]),
}

# Short hyperparameter descriptions for the comparison table
hyperparam_summary = {
    "Logistic Regression": "C=1.0, max_iter=1000",
    "K-Nearest Neighbors": "n_neighbors=5, weights='uniform'",
    "Decision Tree": "max_depth=8",
}

Numeric features: ['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
Categorical features: ['Gender', 'Customer Type', 'Type of Travel', 'Class']


C:\Users\rizos\AppData\Local\Temp\ipykernel_22112\2926097321.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()


## 5. Fit Each Candidate, Time it, Score it

In this section, each candidate pipeline is fitted on the same training data and evaluated on the same test data.

For each model, we calculate:

- fitting time,
- test accuracy,
- test F1-score.

The F1-score is calculated using `satisfied` as the positive class. The comparison table will later be saved as `model_comparison.csv` and displayed in the Gradio app.

In [5]:
rows = []
fitted = {}

for name, pipe in candidates.items():
    t0 = time.perf_counter()
    
    # Fit the pipeline
    pipe.fit(X_train, y_train)
    
    fit_time = time.perf_counter() - t0
    
    # Predict on the test set
    y_pred = pipe.predict(X_test)
    
    # Store model results
    rows.append({
        "Algorithm": name,
        "Hyperparameter": hyperparam_summary[name],
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1": round(f1_score(y_test, y_pred, pos_label=POSITIVE_LABEL), 4),
        "Fit time (s)": round(fit_time, 3),
    })
    
    # Store fitted pipeline
    fitted[name] = pipe

comparison = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)

comparison

,Algorithm,Hyperparameter,Accuracy,F1,Fit time (s)
0,Decision Tree,max_depth=8,0.928,0.9153,0.040
1,K-Nearest Neighbors,"n_neighbors=5, weights='uniform'",0.906,0.8845,0.051
2,Logistic Regression,"C=1.0, max_iter=1000",0.884,0.8606,0.083


## 6. Persist Αrtifacts

We persist two files for the app to read at startup:

- `model.joblib`, which contains the fitted candidate pipelines and useful metadata,
- `model_comparison.csv`, which contains the offline test-set scores for each candidate model.

The Gradio app will load these files and will not train any model at runtime.

In [6]:
winner_name = comparison.iloc[0]["Algorithm"]

print(f"Winner: {winner_name} (F1 = {comparison.iloc[0]['F1']})")

WINNER_JUSTIFICATION = (
    f"{winner_name} achieved the highest F1-score among the three candidate "
    "classification pipelines on the same train/test split. For this reason, "
    "it was selected as the default model for deployment in the Gradio app. "
    "The advanced app uses a smaller bundled sample dataset for deployment purposes, "
    "so the model ranking may differ from the full classification notebook."
)

joblib.dump({
    "pipelines": fitted,
    "feature_names": list(X.columns),
    "target_name": TARGET,
    "positive_label": POSITIVE_LABEL,
    "negative_label": NEGATIVE_LABEL,
    "winner_name": winner_name,
    "justification": WINNER_JUSTIFICATION,
}, HERE / "model.joblib")

comparison.to_csv(HERE / "model_comparison.csv", index=False)

print("saved model.joblib and model_comparison.csv")

Winner: Decision Tree (F1 = 0.9153)
saved model.joblib and model_comparison.csv


## 7. Sanity Check — Reload, Predict, Classification Report

Always reload the saved `model.joblib` file and verify that prediction still works.

If this section runs successfully, it is a good indication that the same saved artifact can also be loaded by `app.py`.

In [7]:
reloaded = joblib.load(HERE / "model.joblib")

print("pipelines bundled:", list(reloaded["pipelines"]))
print("default winner:", reloaded["winner_name"])

pipelines bundled: ['Logistic Regression', 'K-Nearest Neighbors', 'Decision Tree']
default winner: Decision Tree


In [8]:
one_row = X_test.iloc[[0]]

print("One test observation:")
display(one_row)

print("\nPredictions from each fitted pipeline:")

for name, pipe in reloaded["pipelines"].items():
    pred = pipe.predict(one_row)[0]
    proba = pipe.predict_proba(one_row)[0]
    classes = pipe.classes_
    
    print(f"\n{name}")
    print("prediction:", pred)
    
    for class_name, probability in zip(classes, proba):
        print(f"{class_name}: {probability:.3f}")

One test observation:


,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,...,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes
4497,Male,Loyal Customer,37,Business travel,Business,3719,4,4,4,4,...,4,5,5,5,5,5,5,5,0,0.0



Predictions from each fitted pipeline:

Logistic Regression
prediction: satisfied
neutral or dissatisfied: 0.016
satisfied: 0.984

K-Nearest Neighbors
prediction: satisfied
neutral or dissatisfied: 0.000
satisfied: 1.000

Decision Tree
prediction: satisfied
neutral or dissatisfied: 0.005
satisfied: 0.995


In [9]:
winner_pipe = reloaded["pipelines"][reloaded["winner_name"]]

print(classification_report(
    y_test,
    winner_pipe.predict(X_test),
    labels=[NEGATIVE_LABEL, POSITIVE_LABEL],
    target_names=[NEGATIVE_LABEL, POSITIVE_LABEL]
))

                         precision    recall  f1-score   support

neutral or dissatisfied       0.92      0.95      0.94       567
              satisfied       0.93      0.90      0.92       433

               accuracy                           0.93      1000
              macro avg       0.93      0.92      0.93      1000
           weighted avg       0.93      0.93      0.93      1000



## 8. How the Bundled Sample was Created

The deployed Gradio app uses a smaller bundled sample of the original classification training dataset.

The full dataset is not needed at runtime, because the app only needs:

- the saved fitted model bundle (`model.joblib`),
- the offline model comparison table (`model_comparison.csv`),
- a lightweight sample dataset for demonstration and exploratory display.

The code below shows how the bundled sample file was originally created from the full training dataset. It is provided for reproducibility purposes only and does not need to be executed every time the notebook runs.

For the deployed app, only the smaller file `data/airline_passenger_satisfaction_sample.csv` is required.

In [10]:
# This cell documents how the bundled sample was created.
# It is intentionally commented out because the app does not need the full original dataset at runtime.
#
# FULL_DATA_PATH = HERE / "data" / "classification_dataset_train.csv"
# SAMPLE_DATA_PATH = HERE / "data" / "airline_passenger_satisfaction_sample.csv"
#
# full_df = pd.read_csv(FULL_DATA_PATH)
#
# sample_df, _ = train_test_split(
#     full_df,
#     train_size=5000,
#     stratify=full_df["satisfaction"],
#     random_state=42
# )
#
# sample_df.to_csv(SAMPLE_DATA_PATH, index=False)
#
# print("Bundled sample created.")
# print("Sample shape:", sample_df.shape)
# print("Saved to:", SAMPLE_DATA_PATH)